# Scikit-Learn Learning Guide
A hands-on tour of the most important Scikit-Learn features, grouped by topic.

## Contents

|   | Topic | Key APIs |
|---|-------|----------|
| 1 | [Import & Setup](#1.-Import-&-Setup) | `sklearn`, datasets, versioning |
| 2 | [Built-in Datasets](#2.-Built-in-Datasets) | `load_iris`, `load_diabetes`, `make_classification` |
| 3 | [Train / Test Split](#3.-Train-/-Test-Split) | `train_test_split`, stratify |
| 4 | [Preprocessing](#4.-Preprocessing) | `StandardScaler`, `MinMaxScaler`, `LabelEncoder`, `OneHotEncoder`, `SimpleImputer` |
| 5 | [Linear Models](#5.-Linear-Models) | `LinearRegression`, `Ridge`, `Lasso`, `LogisticRegression` |
| 6 | [Tree-Based Models](#6.-Tree-Based-Models) | `DecisionTreeClassifier`, `RandomForestClassifier`, `GradientBoostingClassifier` |
| 7 | [Support Vector Machines](#7.-Support-Vector-Machines) | `SVC`, `SVR`, `LinearSVC` |
| 8 | [k-Nearest Neighbors](#8.-k-Nearest-Neighbors) | `KNeighborsClassifier`, `KNeighborsRegressor` |
| 9 | [Clustering](#9.-Clustering) | `KMeans`, `DBSCAN`, `AgglomerativeClustering` |
| 10 | [Dimensionality Reduction](#10.-Dimensionality-Reduction) | `PCA`, `TruncatedSVD` |
| 11 | [Model Evaluation](#11.-Model-Evaluation) | `accuracy_score`, `confusion_matrix`, `classification_report`, `cross_val_score` |
| 12 | [Pipelines](#12.-Pipelines) | `Pipeline`, `ColumnTransformer`, `make_pipeline` |
| 13 | [Hyperparameter Tuning](#13.-Hyperparameter-Tuning) | `GridSearchCV`, `RandomizedSearchCV` |
| 14 | [Feature Importance & Selection](#14.-Feature-Importance-&-Selection) | `feature_importances_`, `SelectKBest`, `RFE` |

## 1. Import & Setup

In [ ]:
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("scikit-learn version:", sklearn.__version__)
print("numpy version:       ", np.__version__)
print("pandas version:      ", pd.__version__)

import warnings
warnings.filterwarnings('ignore')

## 2. Built-in Datasets

Scikit-learn ships with toy datasets (`load_*`) and synthetic generators (`make_*`).

In [ ]:
from sklearn.datasets import load_iris, load_diabetes

# Classification: Iris (150 samples, 4 features, 3 classes)
iris = load_iris(as_frame=True)
print("=== Iris ===")
print(iris.frame.head())
print("Classes:", iris.target_names)
print("Features:", iris.feature_names)

# Regression: Diabetes (442 samples, 10 features)
diabetes = load_diabetes(as_frame=True)
print("\n=== Diabetes ===")
print(diabetes.frame.describe().round(2))

In [ ]:
from sklearn.datasets import make_classification, make_regression, make_blobs
import numpy as np
import matplotlib.pyplot as plt

# Synthetic classification
X_cls, y_cls = make_classification(
    n_samples=300, n_features=4, n_informative=2,
    n_redundant=1, n_classes=2, random_state=42
)
print("Classification X shape:", X_cls.shape, "| y unique:", np.unique(y_cls))

# Synthetic regression
X_reg, y_reg = make_regression(n_samples=200, n_features=5, noise=15.0, random_state=42)
print("Regression    X shape:", X_reg.shape)

# Blobs for clustering
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
print("Blobs         X shape:", X_blobs.shape, "| centers:", np.unique(y_blobs))

plt.figure(figsize=(5, 3))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='tab10', s=15)
plt.title("make_blobs – 4 clusters")
plt.tight_layout()
plt.show()

## 3. Train / Test Split

Always split **before** any fitting to prevent data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

# 80% train / 20% test, stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)
print("Train class dist:", y_train.value_counts().to_dict())
print("Test  class dist:", y_test.value_counts().to_dict())

# Three-way split: 60 / 20 / 20
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)
print(f"\nTrain: {len(X_tr)}  Val: {len(X_val)}  Test: {len(X_te)}")

## 4. Preprocessing

Fit scalers/encoders **only on training data**, then transform both train and test.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
import numpy as np

X_raw = np.array([[100, 0.01], [200, 0.02], [150, 0.015], [300, 0.03]])

# StandardScaler: zero mean, unit variance
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X_raw)
print("StandardScaler (mean~0, std~1):")
print(X_std.round(3))

# MinMaxScaler: scales to [0, 1]
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(X_raw)
print("\nMinMaxScaler (range [0,1]):")
print(X_mm.round(3))

# RobustScaler: uses median + IQR, robust to outliers
scaler_rob = RobustScaler()
X_rob = scaler_rob.fit_transform(X_raw)
print("\nRobustScaler (median/IQR):")
print(X_rob.round(3))

In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
import numpy as np

# LabelEncoder: string classes -> integers (for target y)
le = LabelEncoder()
y_str = ['cat', 'dog', 'cat', 'bird', 'dog']
y_enc = le.fit_transform(y_str)
print("LabelEncoder:")
print(" classes:", le.classes_)
print(" encoded:", y_enc)
print(" decoded:", le.inverse_transform(y_enc))

# OrdinalEncoder: ordered categories in feature columns (2-D)
oe = OrdinalEncoder(categories=[['low', 'medium', 'high']])
X_ord = np.array([['low'], ['high'], ['medium'], ['low']])
print("\nOrdinalEncoder:", oe.fit_transform(X_ord).ravel())

# OneHotEncoder: nominal categories (unordered)
ohe = OneHotEncoder(sparse_output=False)
X_cat = np.array([['cat'], ['dog'], ['cat'], ['bird']])
X_ohe = ohe.fit_transform(X_cat)
print("\nOneHotEncoder categories:", ohe.categories_)
print(X_ohe)

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np

X_missing = np.array([
    [1.0, 2.0,  np.nan],
    [3.0, np.nan, 4.0],
    [np.nan, 6.0, 7.0],
    [8.0, 9.0, 10.0],
])

# Mean imputation
imp_mean = SimpleImputer(strategy='mean')
print("Mean imputation:")
print(imp_mean.fit_transform(X_missing))

# Median imputation (better with outliers)
imp_median = SimpleImputer(strategy='median')
print("\nMedian imputation:")
print(imp_median.fit_transform(X_missing))

# Most frequent for categorical
X_cat_missing = np.array([['a'], ['b'], [None], ['a'], [None]])
imp_freq = SimpleImputer(strategy='most_frequent')
print("\nMost-frequent:", imp_freq.fit_transform(X_cat_missing).ravel())

## 5. Linear Models

Fast, interpretable baselines. Always a good starting point.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X, y = make_regression(n_samples=300, n_features=10, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge(alpha=1)':   Ridge(alpha=1.0),
    'Lasso(alpha=0.5)': Lasso(alpha=0.5),
    'ElasticNet':       ElasticNet(alpha=0.5, l1_ratio=0.5),
}

print(f"{'Model':<25} {'RMSE':>8} {'R2':>8}")
print("-" * 43)
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2   = r2_score(y_test, preds)
    print(f"{name:<25} {rmse:>8.2f} {r2:>8.4f}")

lr = LinearRegression().fit(X_train, y_train)
print("\nCoefficients:", lr.coef_.round(2))
print("Intercept:   ", round(lr.intercept_, 2))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

# Always scale before LogisticRegression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, C=1.0)   # C = inverse regularization strength
lr.fit(X_train_s, y_train)

print("Accuracy:", accuracy_score(y_test, lr.predict(X_test_s)))

proba = lr.predict_proba(X_test_s[:5])
print("\nClass probabilities (first 5 test samples):")
for true, p in zip(y_test[:5], proba):
    print(f"  True: {iris.target_names[true]:<12} probs: {p.round(3)}")

## 6. Tree-Based Models

Decision trees, random forests, and gradient boosting — the workhorses of tabular ML.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
print("DecisionTree accuracy:", accuracy_score(y_test, dt.predict(X_test)))
print("\nTree rules:")
print(export_text(dt, feature_names=list(iris.feature_names)))

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

cancer = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target
)

ensembles = {
    'RandomForest':     RandomForestClassifier(n_estimators=100, random_state=42),
    'ExtraTrees':       ExtraTreesClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
}

print(f"{'Model':<22} {'Accuracy':>9}")
print("-" * 33)
for name, model in ensembles.items():
    model.fit(X_tr, y_tr)
    print(f"{name:<22} {accuracy_score(y_te, model.predict(X_te)):>9.4f}")

# Feature importances
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
importances = rf.feature_importances_
top_idx = np.argsort(importances)[::-1][:5]
print("\nTop-5 features (RandomForest):")
for i in top_idx:
    print(f"  {cancer.feature_names[i]:<35} {importances[i]:.4f}")

## 7. Support Vector Machines

SVMs find the maximum-margin hyperplane. Powerful for high-dimensional or small datasets. **Always scale first.**

In [ ]:
from sklearn.svm import SVC, SVR
from sklearn.datasets import load_iris, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

# SVC: classification
iris = load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

for kernel in ['linear', 'rbf', 'poly']:
    svc = SVC(kernel=kernel, C=1.0, random_state=42)
    svc.fit(X_tr_s, y_tr)
    acc = accuracy_score(y_te, svc.predict(X_te_s))
    sv  = svc.support_vectors_.shape[0]
    print(f"SVC kernel={kernel:<8} accuracy={acc:.4f}  support_vectors={sv}")

# SVR: regression
X_r, y_r = make_regression(n_samples=200, n_features=5, noise=10, random_state=42)
X_rtr, X_rte, y_rtr, y_rte = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
sc = StandardScaler()
svr = SVR(kernel='rbf', C=100, epsilon=0.1)
svr.fit(sc.fit_transform(X_rtr), y_rtr)
rmse = mean_squared_error(y_rte, svr.predict(sc.transform(X_rte))) ** 0.5
print(f"\nSVR (rbf) RMSE: {rmse:.2f}")

## 8. k-Nearest Neighbors

Non-parametric: predictions based on K closest training points. No training cost, but slow at inference.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

iris = load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

print("k    Accuracy")
print("-" * 20)
for k in [1, 3, 5, 7, 11, 15]:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_tr_s, y_tr)
    acc = accuracy_score(y_te, knn.predict(X_te_s))
    print(f"{k:<5}{acc:.4f}")

# Plot accuracy vs k
ks   = range(1, 21)
accs = [accuracy_score(y_te, KNeighborsClassifier(n_neighbors=k).fit(X_tr_s, y_tr).predict(X_te_s)) for k in ks]
plt.figure(figsize=(7, 3))
plt.plot(ks, accs, marker='o')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.title('KNN Accuracy vs. k (Iris)')
plt.xticks(list(ks))
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Clustering

Unsupervised: discover natural groupings without labels.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.7, random_state=42)

km = KMeans(n_clusters=4, n_init=10, random_state=42)
km.fit(X)
labels = km.labels_

print("Inertia (within-cluster SSE):", round(km.inertia_, 2))
print("Silhouette score:            ", round(silhouette_score(X, labels), 4))
print("Cluster sizes:", dict(zip(*np.unique(labels, return_counts=True))))

# Elbow method to choose k
inertias    = []
silhouettes = []
for k in range(2, 9):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(X, km_k.labels_))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(range(2, 9), inertias, marker='o')
axes[0].set_title('Elbow – Inertia'); axes[0].set_xlabel('k')
axes[1].plot(range(2, 9), silhouettes, marker='s', color='orange')
axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('k')
plt.tight_layout(); plt.show()

# Visualise final clusters
plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=15, alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='black', marker='X', s=150, label='Centroids')
plt.title('KMeans Clusters (k=4)'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# DBSCAN handles non-convex shapes and automatically finds outliers (-1)
X_moon, _ = make_moons(n_samples=300, noise=0.05, random_state=42)
X_moon = StandardScaler().fit_transform(X_moon)

db = DBSCAN(eps=0.3, min_samples=5)
labels_db = db.fit_predict(X_moon)

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
print(f"DBSCAN: {n_clusters} clusters, {(labels_db == -1).sum()} noise points")

# Agglomerative (hierarchical) clustering
agg = AgglomerativeClustering(n_clusters=2, linkage='ward')
labels_agg = agg.fit_predict(X_moon)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X_moon[:, 0], X_moon[:, 1], c=labels_db, cmap='tab10', s=15)
axes[0].set_title('DBSCAN')
axes[1].scatter(X_moon[:, 0], X_moon[:, 1], c=labels_agg, cmap='tab10', s=15)
axes[1].set_title('Agglomerative (ward, k=2)')
plt.tight_layout(); plt.show()

## 10. Dimensionality Reduction

Reduce features while retaining the most information — useful for visualization and noise removal.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

digits = load_digits()   # 1797 samples, 64 features (8x8 pixel images)
X_d = StandardScaler().fit_transform(digits.data)

# How many components explain 95% of variance?
pca_full = PCA().fit(X_d)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n95 = np.searchsorted(cumvar, 0.95) + 1
print(f"Components to explain 95% variance: {n95} (out of {X_d.shape[1]})")

plt.figure(figsize=(7, 3))
plt.plot(cumvar * 100)
plt.axhline(95, color='red', linestyle='--', label='95%')
plt.axvline(n95, color='orange', linestyle='--', label=f'n={n95}')
plt.xlabel('Number of components'); plt.ylabel('Cumulative variance (%)')
plt.title('PCA – Digits'); plt.legend(); plt.tight_layout(); plt.show()

# 2D projection for visualization
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_d)

plt.figure(figsize=(7, 5))
sc = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=digits.target, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(sc, ticks=range(10), label='Digit')
plt.title('PCA – 2D projection of Digits')
plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.tight_layout(); plt.show()

## 11. Model Evaluation

Choose the right metric for the task. Never rely on accuracy alone for imbalanced datasets.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

cancer = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
y_pred  = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]

print(f"Accuracy : {accuracy_score(y_te, y_pred):.4f}")
print(f"Precision: {precision_score(y_te, y_pred):.4f}")
print(f"Recall   : {recall_score(y_te, y_pred):.4f}")
print(f"F1       : {f1_score(y_te, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_te, y_proba):.4f}")

# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred 0', 'Pred 1'])
ax.set_yticklabels(['True 0', 'True 1'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=14)
plt.title('Confusion Matrix'); plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

print("\nClassification Report:")
print(classification_report(y_te, y_pred, target_names=cancer.target_names))

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()
rf   = RandomForestClassifier(n_estimators=100, random_state=42)
cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 5-fold cross-validation
scores = cross_val_score(rf, iris.data, iris.target, cv=cv, scoring='accuracy')
print("5-Fold CV Accuracy:")
print(f"  Scores: {scores.round(4)}")
print(f"  Mean:   {scores.mean():.4f} (+/- {scores.std():.4f})")

# Multiple metrics at once
results = cross_validate(
    rf, iris.data, iris.target,
    cv=cv,
    scoring=['accuracy', 'f1_weighted'],
    return_train_score=True
)
print("\ncross_validate results:")
for key, val in results.items():
    print(f"  {key:<30} mean={val.mean():.4f}  std={val.std():.4f}")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

diabetes = load_diabetes()
X_tr, X_te, y_tr, y_te = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42
)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_tr, y_tr)
y_pred = rf_reg.predict(X_te)

print("Regression Metrics:")
print(f"  MAE  : {mean_absolute_error(y_te, y_pred):.2f}")
print(f"  RMSE : {mean_squared_error(y_te, y_pred)**0.5:.2f}")
print(f"  R2   : {r2_score(y_te, y_pred):.4f}")

## 12. Pipelines

`Pipeline` chains preprocessing + model into a single leak-proof object. Always prefer pipelines.

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(
    digits.data, digits.target, test_size=0.2, random_state=42, stratify=digits.target
)

# Named Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=30)),
    ('clf',    SVC(kernel='rbf', C=10)),
])
pipe.fit(X_tr, y_tr)
print("Pipeline accuracy:", pipe.score(X_te, y_te))

# make_pipeline shorthand (auto-names steps)
pipe2 = make_pipeline(StandardScaler(), PCA(n_components=30), SVC(kernel='rbf', C=10))
print("Step names:", list(pipe2.named_steps.keys()))

# Pipelines work transparently with cross_val_score
scores = cross_val_score(pipe, digits.data, digits.target, cv=5, scoring='accuracy')
print(f"5-fold CV: mean={scores.mean():.4f}  std={scores.std():.4f}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# Mixed numeric + categorical dataset
np.random.seed(42)
N = 200
df = pd.DataFrame({
    'age':    np.random.randint(18, 65, N).astype(float),
    'salary': np.random.normal(50000, 15000, N),
    'city':   np.random.choice(['Tehran', 'London', 'Tokyo', None], N),
    'gender': np.random.choice(['M', 'F'], N),
})
df.loc[np.random.choice(N, 20, replace=False), 'age']    = np.nan
df.loc[np.random.choice(N, 15, replace=False), 'salary'] = np.nan
y = np.random.randint(0, 2, N)

X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.2, random_state=42)

num_pipe = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                     ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_pipe, ['age', 'salary']),
    ('cat', cat_pipe, ['city', 'gender']),
])

full_pipe = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
full_pipe.fit(X_tr, y_tr)
print("ColumnTransformer + RandomForest accuracy:", accuracy_score(y_te, full_pipe.predict(X_te)))

## 13. Hyperparameter Tuning

Search the parameter space systematically. Always tune inside a pipeline to avoid leakage.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.datasets import load_iris

iris = load_iris()
pipe = make_pipeline(StandardScaler(), SVC())

# Keys: step_name__parameter_name
param_grid = {
    'svc__C':      [0.1, 1, 10, 100],
    'svc__kernel': ['linear', 'rbf'],
    'svc__gamma':  ['scale', 'auto'],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(iris.data, iris.target)

print("Best params:",    grid.best_params_)
print("Best CV score:",  round(grid.best_score_, 4))
print("Best estimator:", grid.best_estimator_)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from scipy.stats import randint, uniform
import pandas as pd

cancer = load_breast_cancer()

# Continuous distributions sample the space more efficiently than grids
param_dist = {
    'n_estimators':      randint(50, 300),
    'max_depth':         randint(3, 15),
    'min_samples_split': randint(2, 20),
    'max_features':      uniform(0.1, 0.9),
}

rf = RandomForestClassifier(random_state=42)
rand_search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=30, cv=5, scoring='roc_auc',
    n_jobs=-1, random_state=42, verbose=1
)
rand_search.fit(cancer.data, cancer.target)

print("Best params:",  rand_search.best_params_)
print("Best ROC-AUC:", round(rand_search.best_score_, 4))

results_df = pd.DataFrame(rand_search.cv_results_)
print("\nTop-5 combinations:")
print(
    results_df.sort_values('rank_test_score')
    [['param_n_estimators', 'param_max_depth', 'mean_test_score']]
    .head().to_string(index=False)
)

## 14. Feature Importance & Selection

Identify informative features to simplify models, speed up training, and improve generalization.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
feat_names = cancer.feature_names

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. Random Forest feature importances
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(8, 3))
plt.bar(range(10), importances[idx])
plt.xticks(range(10), feat_names[idx], rotation=45, ha='right', fontsize=8)
plt.title('RandomForest — Top 10 Feature Importances')
plt.tight_layout(); plt.show()

# 2. SelectKBest (univariate F-test)
selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X_tr, y_tr)
print("SelectKBest top-10 features:")
print(feat_names[selector.get_support()].tolist())
X_tr_k = selector.transform(X_tr)
X_te_k = selector.transform(X_te)
acc_k = accuracy_score(y_te, RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr_k, y_tr).predict(X_te_k))
print(f"  Accuracy with 10 features: {acc_k:.4f}")

# 3. Recursive Feature Elimination (RFE)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

lr = LogisticRegression(max_iter=5000)
rfe = RFE(estimator=lr, n_features_to_select=10, step=1)
rfe.fit(X_tr_s, y_tr)
print("\nRFE selected features:")
print(feat_names[rfe.support_].tolist())
print(f"  Accuracy with RFE 10 features: {accuracy_score(y_te, rfe.predict(X_te_s)):.4f}")
print(f"  Accuracy with all 30 features: {accuracy_score(y_te, lr.fit(X_tr_s, y_tr).predict(X_te_s)):.4f}")